In [7]:
from transformer_lens.model_bridge import TransformerBridge
import os
from dotenv import load_dotenv
load_dotenv()
# HF_KEY = os.getenv("HF_KEY")

# Method 1: Simple loading (recommended)
bridge = TransformerBridge.boot_transformers(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    # token=HF_KEY,
    device="cpu",  # or "cpu"
    # dtype="float32",
    trust_remote_code=False,  # Usually not needed for official models
)

# Test basic forward pass
# text = "Hello, how are you?"
# logits = bridge(text, return_type="logits")
# print(f"Logits shape: {logits.shape}")

# Test text generation
generated = bridge.generate("Once upon a time", max_new_tokens=50)
print(f"Generated text: {generated}")

# # Test run_with_cache (for mechanistic interpretability)
# logits, cache = bridge.run_with_cache("The meaning of life is")
# print(f"Cache keys: {cache.keys()}")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:21<00:00,  2.31it/s]

Generated text: Once upon a time, there were three kingdoms: West Kingdom, North Kingdom, and East Kingdom.

Each kingdom had its own culture and leadership structure. West Kingdom had a single king, who because of his militaristic background, had his likes turned into his dislikes. The


In [24]:
import os
import json
from pathlib import Path
from collections import Counter
import pandas as pd
base_soln_dir = r"C:\Users\Preet Lodaya\Thought_Anchors\thought-anchors\math_rollouts\accounts\pdlodaya-l7vcn0oxxgj\deployments\dmmz204v\temperature_0.6_top_p_0.95\correct_base_solution"

# Collect all is_correct values
results_df = pd.DataFrame()
row_dict = {}
# Iterate through all subdirectories
for folder in os.listdir(base_soln_dir):
    # print(folder)
    folder_path = os.path.join(base_soln_dir, folder)
    
    # Check if it's a directory
    if os.path.isdir(folder_path):
        base_solution_file = os.path.join(folder_path, "base_solution.json")
        problem_file = os.path.join(folder_path, "problem.json")
        # Check if base_solution.json exists
        if os.path.exists(base_solution_file):
            try:
                with open(base_solution_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    
                row_dict['is_correct'] = data['is_correct']
                row_dict['model_answer'] = data['answer']
                row_dict['full_cot'] = data['full_cot']
            except Exception as e:
                print(f"Error reading {base_solution_file}: {e}")
        if os.path.exists(base_solution_file):
            with open(problem_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            row_dict['level'] = data['level']
            row_dict['type'] = data['type']
            row_dict['correct_answer'] = data['gt_answer']
            row_dict['problem_id'] = folder_path.split("\\")[-1]
        results_df = pd.concat([results_df, pd.DataFrame([row_dict])], ignore_index=True)
# Count the values
print(results_df['is_correct'].value_counts())

is_correct
True     380
False     43
Name: count, dtype: int64


In [25]:
results_df.groupby('level')['is_correct'].value_counts()

level    is_correct
Level 1  True           57
         False           2
Level 2  True          195
         False          21
Level 3  True           39
         False          11
Level 4  True           44
         False           3
Level 5  True           45
         False           6
Name: count, dtype: int64

In [30]:
results_df.groupby('type')['is_correct'].value_counts()

type                    is_correct
Algebra                 True          104
                        False          12
Counting & Probability  True           31
                        False           3
Geometry                True           34
                        False           8
Intermediate Algebra    True           50
                        False          10
Number Theory           True           38
                        False           2
Prealgebra              True           76
                        False           4
Precalculus             True           47
                        False           4
Name: count, dtype: int64

In [28]:
results_df.loc[results_df['is_correct'] == False].sort_values('level')

,is_correct,model_answer,full_cot,level,type,correct_answer,problem_id
191,False,32768,Solve this math problem step by step. You MUST...,Level 1,Intermediate Algebra,32,problem_4403
231,False,1,Solve this math problem step by step. You MUST...,Level 1,Number Theory,3,problem_5056
54,False,425,Solve this math problem step by step. You MUST...,Level 2,Algebra,\$425,problem_1658
55,False,425,Solve this math problem step by step. You MUST...,Level 2,Algebra,\$425,problem_1658
56,False,"(-2,\ 3)",Solve this math problem step by step. You MUST...,Level 2,Algebra,"(-2,3)",problem_1677
57,False,"(-2,\ 3)",Solve this math problem step by step. You MUST...,Level 2,Algebra,"(-2,3)",problem_1677
97,False,9,Solve this math problem step by step. You MUST...,Level 2,Geometry,36,problem_2536
110,False,55,Solve this math problem step by step. You MUST...,Level 2,Geometry,55^\circ,problem_2718
118,False,"256\,\pi\ \text{cubic feet}",Solve this math problem step by step. You MUST...,Level 2,Geometry,256\pi,problem_2808
142,False,0.3,Solve this math problem step by step. You MUST...,Level 2,Algebra,.3,problem_342


In [32]:
results_df.loc[results_df['type'] == "Geometry"].sort_values('level').drop_duplicates()

,is_correct,model_answer,full_cot,level,type,correct_answer,problem_id
100,True,20,Solve this math problem step by step. You MUST...,Level 1,Geometry,20,problem_2606
122,True,17,Solve this math problem step by step. You MUST...,Level 1,Geometry,17,problem_2900
132,True,0,Solve this math problem step by step. You MUST...,Level 1,Geometry,0,problem_3252
97,False,9,Solve this math problem step by step. You MUST...,Level 2,Geometry,36,problem_2536
102,True,\dfrac{\sqrt{3}}{2},Solve this math problem step by step. You MUST...,Level 2,Geometry,\frac{\sqrt{3}}{2},problem_2617
103,True,12,Solve this math problem step by step. You MUST...,Level 2,Geometry,12,problem_2629
104,True,78\pi,Solve this math problem step by step. You MUST...,Level 2,Geometry,78\pi,problem_2632
110,False,55,Solve this math problem step by step. You MUST...,Level 2,Geometry,55^\circ,problem_2718
116,True,30,Solve this math problem step by step. You MUST...,Level 2,Geometry,30,problem_2779
118,False,"256\,\pi\ \text{cubic feet}",Solve this math problem step by step. You MUST...,Level 2,Geometry,256\pi,problem_2808


In [22]:
results_df.iloc[51]['full_cot']

"Solve this math problem step by step. You MUST put your final answer in \\boxed{}. Problem: The coefficients of the polynomial\n\\[x^4 + bx^3 + cx^2 + dx + e = 0\\]are all integers.  Let $n$ be the exact number of integer roots of the polynomial, counting multiplicity.  For example, the polynomial $(x + 3)^2 (x^2 + 4x + 11) = 0$ has two integer roots counting multiplicity, because the root $-3$ is counted twice.\n\nEnter all possible values of $n,$ separated by commas. Solution: \n<think>\nOkay, so I have this problem where I need to find all possible values of n, which is the exact number of integer roots of a polynomial with integer coefficients. The polynomial is a quartic, meaning it's degree 4, and it looks like this:\n\n\\[x^4 + bx^3 + cx^2 + dx + e = 0\\]\n\nAll the coefficients b, c, d, e are integers. The question is about the number of integer roots, counting multiplicity, which is denoted by n. They gave an example where the polynomial \\((x + 3)^2 (x^2 + 4x + 11) = 0\\) ha

In [21]:
(3*7*2) +(3*5*2)+(3*5*2*2) + (3*2*2*2*2)

180

In [19]:
(3*7*2)*(3*5*2)*(3*5*2)*(3*2*2*2*2)

1814400

In [20]:
10*9*8*7*6*5*4*3*2

3628800

In [ ]:
# [5104, 6765, 7162, 1923, 2542, 4656, 3011, 3558, 489, 3009]

## 

Chunk 0 (base solution) - multiple Rollout Accuracy

In [41]:
for folder in os.listdir(base_soln_dir):
    if 'chunk_0' in os.listdir(os.path.join(base_soln_dir, folder)):
        with open(os.path.join(base_soln_dir, folder, 'chunk_0', 'solutions.json'), 'r') as f:
            solutions = json.load(f)
        print(folder)
        sol_list = [sol['is_correct'] for sol in solutions if 'is_correct' in sol]
        print('Number of solutions:', len(sol_list))
        print(pd.Series(sol_list).value_counts(normalize=True))


problem_1923
Number of solutions: 75
True     0.933333
False    0.066667
Name: proportion, dtype: float64
problem_2542
Number of solutions: 75
True     0.973333
False    0.026667
Name: proportion, dtype: float64
problem_3009
Number of solutions: 0
Series([], Name: proportion, dtype: float64)
problem_3011
Number of solutions: 0
Series([], Name: proportion, dtype: float64)
problem_3558
Number of solutions: 0
Series([], Name: proportion, dtype: float64)
problem_4656
Number of solutions: 0
Series([], Name: proportion, dtype: float64)
problem_489
Number of solutions: 75
True     0.986667
False    0.013333
Name: proportion, dtype: float64
problem_5104
Number of solutions: 0
Series([], Name: proportion, dtype: float64)
problem_6765
Number of solutions: 0
Series([], Name: proportion, dtype: float64)
problem_7162
Number of solutions: 75
True    1.0
Name: proportion, dtype: float64


In [37]:
os.path.join(base_soln_dir,'p\\chunk_0')

'C:\\Users\\Preet Lodaya\\Thought_Anchors\\thought-anchors\\math_rollouts\\accounts\\pdlodaya-l7vcn0oxxgj\\deployments\\dmmz204v\\temperature_0.6_top_p_0.95\\correct_base_solution\\p\\chunk_0'

In [8]:
import os, json

with open("math_rollouts/accounts/pdlodaya-l7vcn0oxxgj/deployments/e2zlc2rg/temperature_0.6_top_p_0.95/correct_base_solution/problem_1923/chunks.json", 'r') as f:
    data = json.load(f)

In [ ]:
import pandas as pd
lengths = []
# for chunk in data['chunks']:
    lengths.append(bridge.to_tokens(chunk).shape[-1])
pd.Series(lengths).describe()

count    170.000000
mean      14.011765
std        7.812892
min        3.000000
25%        9.000000
50%       12.000000
75%       18.000000
max       51.000000
dtype: float64

In [13]:
bridge.to_tokens(data['solution_text']).shape

torch.Size([1, 2449])

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

from pathlib import Path
import json


from dotenv import load_dotenv

# PROJECT_ROOT = Path(__file__).resolve().parent[3]
# print(PROJECT_ROOT)

load_dotenv()

def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

RESAMPLED_BASE_DIR = Path(r"C:\Users\Preet Lodaya\Thought_Anchors\thought-anchors\math_rollouts\deepseek-r1-distill-qwen-14b\temperature_0.6_top_p_0.95")
SIMILARITY_MODEL_NAME = "all-MiniLM-L6-v2"
_similarity_model_cache = {}

def get_similarity_model(model_name: str = SIMILARITY_MODEL_NAME):
    if model_name not in _similarity_model_cache:
        _similarity_model_cache[model_name] = SentenceTransformer(model_name)
    return _similarity_model_cache[model_name]

def cosine_similarity(vec_a, vec_b):
    denom = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    return float(np.dot(vec_a, vec_b) / denom) if denom else None

def get_resampled_chunk_similarities(problem_id, chunk_idx, base_dir=RESAMPLED_BASE_DIR, model_name: str = SIMILARITY_MODEL_NAME, solution_type: str = "correct"):
    problem_name = str(problem_id)
    if not problem_name.startswith("problem_"):
        problem_name = f"problem_{problem_name}"

    solutions_path = Path(base_dir) / f"{solution_type}_base_solution" / problem_name / f"chunk_{chunk_idx}" / "solutions.json"
    if not solutions_path.exists():
        raise FileNotFoundError(f"No resampled solutions found at {solutions_path}")

    solutions = load_json(solutions_path)
    model = get_similarity_model(model_name)
    rows = []
    texts_to_embed = []
    pending_pairs = []
    removed_chunk = None

    for solution_idx, sol in enumerate(solutions):
        removed = sol.get("chunk_removed")
        resampled = sol.get("chunk_resampled")

        if removed_chunk is None and isinstance(removed, str):
            removed_chunk = removed

        row = {
            "solution_idx": solution_idx,
            "chunk_resampled": resampled,
            "is_correct": sol.get("is_correct"),
            "answer": sol.get("answer"),
            "similarity": None,
        }
        rows.append(row)

        if isinstance(removed, str) and isinstance(resampled, str):
            pending_pairs.append(solution_idx)
            # texts_to_embed.extend([f"Following statements is from a math solution :- {removed}", f"Following statements is from a math solution :- {resampled}"])
            texts_to_embed.extend([removed, resampled])
    if pending_pairs:
        embeddings = model.encode(texts_to_embed, convert_to_numpy=True)
        for pair_idx, solution_idx in enumerate(pending_pairs):
            removed_embedding = embeddings[2 * pair_idx]
            resampled_embedding = embeddings[2 * pair_idx + 1]
            rows[solution_idx]["similarity"] = cosine_similarity(removed_embedding, resampled_embedding)

    results_df = pd.DataFrame(rows)
    valid_similarities = results_df["similarity"].dropna().tolist()

    print("removed_chunk:")
    print(removed_chunk)

    summary = {
        "problem_id": problem_name,
        "chunk_idx": int(chunk_idx),
        "solutions_path": str(solutions_path),
        "num_resamples": len(rows),
        "num_valid_pairs": len(valid_similarities),
        "mean_similarity": float(np.mean(valid_similarities)) if valid_similarities else None,
        "min_similarity": float(np.min(valid_similarities)) if valid_similarities else None,
        "max_similarity": float(np.max(valid_similarities)) if valid_similarities else None,
    }
    return removed_chunk, results_df, summary

# removed_chunk, similarity_df, summary = get_resampled_chunk_similarities("problem_330", 33)
# summary

In [25]:
i = 0
similarity_df.sort_values('similarity')['chunk_resampled'].values[i]

'Now, the entire expression is 3*(Layer 9), so I need to compute that: Final result: 3*29524 = Let me calculate that.'

In [26]:
similarity_df.sort_values('similarity')['similarity'].values[i]

np.float64(0.7082080245018005)

In [45]:
removed_chunk, similarity_df, summary = get_resampled_chunk_similarities("problem_330", 72, solution_type="incorrect")
summary

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


removed_chunk:
Let me try to compute each level again carefully.


{'problem_id': 'problem_330',
 'chunk_idx': 72,
 'solutions_path': 'C:\\Users\\Preet Lodaya\\Thought_Anchors\\thought-anchors\\math_rollouts\\deepseek-r1-distill-qwen-14b\\temperature_0.6_top_p_0.95\\incorrect_base_solution\\problem_330\\chunk_72\\solutions.json',
 'num_resamples': 100,
 'num_valid_pairs': 100,
 'mean_similarity': 0.7018475902080535,
 'min_similarity': 0.5355756878852844,
 'max_similarity': 0.9697916507720947}

In [46]:
# i = 0
for i in range(20):
    print(similarity_df.sort_values('similarity')['chunk_resampled'].values[i])

Let me try again, step by step, carefully: Level 1: (1 + 3) = 4 Level 2: 3*(1 + Level 1) = 3*(1 + 4) = 3*5 = 15 Level 3: 3*(1 + Level 2) = 3*(1 + 15) = 3*16 = 48 Level 4: 3*(1 + Level 3) = 3*(1 + 48) = 3*49 = 147 Level 5: 3*(1 + Level 4) = 3*(1 + 147) = 3*148 = 444 Level 6: 3*(1 + Level 5) = 3*(1 + 444) = 3*445 = 1335 Level 7: 3*(1 + Level 6) = 3*(1 + 1335) = 3*1336 = 4008 Level 8: 3*(1 + Level 7) = 3*(1 + 4008) = 3*4009 = 12027 Level 9: 3*(1 + Level 8) = 3*(1 + 12027) = 3*12028 = 36084 So, after 9 levels, the result is 36084.
Let me try again, perhaps I made an error in multiplication somewhere.
Let me try again, carefully, step by step, to ensure I don't make a mistake.
Let me try again, carefully, step by step, to make sure I don't make a mistake.
Let me check again step by step, perhaps I made an arithmetic error.
Let me try again, step by step, carefully, to ensure I don't make a mistake.
Let me double-check.
Let me check again, perhaps I messed up in the earlier steps.
Let me do 

In [47]:
for i in range(20):
    print(similarity_df.sort_values('similarity')['similarity'].values[i])

0.5355756878852844
0.6340112090110779
0.6420028805732727
0.6449149250984192
0.644969642162323
0.6487084627151489
0.6540930271148682
0.6629245281219482
0.6637643575668335
0.6658450365066528
0.6667183041572571
0.6693230271339417
0.6693231463432312
0.6693231463432312
0.6693231463432312
0.6693231463432312
0.6731745600700378
0.674579918384552
0.6745800375938416
0.6745800375938416


In [48]:
# i = 0
for i in range(20):
    print(similarity_df.sort_values('similarity',ascending=False)['chunk_resampled'].values[i])

Let me try to compute each level correctly, step by step.
Let me try to compute each level again step by step, carefully, to avoid errors.
Let me try to compute each level step by step, carefully, to avoid mistakes.
Let me try again, carefully, step by step, ensuring each level is correct.
Let me try again, carefully, step by step, ensuring each level is correct.
Let me check again, perhaps I made a mistake in the number of levels.
Let me make a table to track each level properly.
Let me check each step again to be precise.
Let me check each step again carefully.
Let me check each step again carefully.
Let me check each step again carefully.
Let me check each step again carefully.
Let me check each step again carefully.
Let me check each step again carefully.
Let me check each step again carefully.
Let me double-check each step carefully.
Let me double-check each step carefully.
Let me double-check each step carefully.
Let me check each step carefully again.
Let me check each step care